# 0. 환경설정
## import

In [18]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

## 함수정의

In [34]:
def draw_lineChart (df, y_val, title, ylabel, is_rank=False):

    autorange =  "reversed" if is_rank else True
    dtick = 1 if is_rank else None
    
    # 1. Plotly 라인 차트 생성
    fig = px.line(
        df, 
        x='b_date', 
        y=y_val, 
        color='movieNm',       # 영화별 색상 구분
        markers=True,          # 데이터 지점에 점 표시
        title=title,
        hover_data=['movieNm', 'b_date', y_val] # 마우스 올렸을 때 출력할 데이터
    )
    
    fig.update_layout(
        yaxis=dict(
            autorange=autorange,          # Y축 반전 (1위가 최상단)
            dtick=dtick,                       # Y축 간격을 1 단위로 정수 표시
            title=f'{ylabel} ({y_val})'
        ),
        xaxis=dict(
            title='날짜 (Date)',
            tickangle=-45                   # 날짜 라벨 45도 회전
        ),
        hovermode="x unified",              # 같은 날짜 위치에 마우스를 올리면 모든 영화 순위를 한 번에 비교
        legend_title_text='영화 제목',
        template='plotly_white',            # 배경을 깔끔한 흰색으로 설정
        width=1000,
        height=600
    )

    # 3. 차트 출력
    fig.show()

# 1. 데이터 호출 및 확인

## 데이터 호출

In [20]:
movie_df = pd.read_csv('../data/pre_processed/movie_info_20260724~20260810.csv')
review_df = pd.read_csv('../data/pre_processed/review_20260724~20260810.csv')
master_df = pd.read_csv('../data/pre_processed/movie_master_20260724~20260810.csv')

## 데이터 확인

### review_df : 리뷰 데이터 모음
 * `id` : 영화 ID
 * `reviewer_name` : 리뷰어 이름
 * `score` : 점수
 * `review` : 리뷰 내용

In [21]:
# 데이터 확인
review_df.info()

# 중복값 확인
review_df.duplicated().sum()
# 결측값 확인
review_df.isna().sum()
# 이상치 확인
# score 범위 1 ~ 10점
review_df['score'].min() # 4
review_df['score'].max() # 9

# 필요 전처리 
# id값 텍스트화 시켜야 함
review_df['id'] = review_df['id'].astype('str')

review_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             48 non-null     int64 
 1   reviewer_name  48 non-null     object
 2   score          48 non-null     int64 
 3   review         48 non-null     object
dtypes: int64(2), object(2)
memory usage: 1.6+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             48 non-null     object
 1   reviewer_name  48 non-null     object
 2   score          48 non-null     int64 
 3   review         48 non-null     object
dtypes: int64(1), object(3)
memory usage: 1.6+ KB


### master_df : 영화의 정보를 담고, 리뷰와의 연결성을 확보할 수 있는 테이블

* `title` : 영화 제목
* `score` : 영화 평점
* `id` : 영화 id
* `genre` : 영화 장르
* `grade` : 등급
* `time` : 영화 상영시간
* `director` : 감독
* `expert_score` : 전문가 평점
* `nation` : 국가
* `movieCd` : 영화코드 (API 데이터 연결용)

In [22]:
# 데이터 확인
master_df.info()

# 데이터 형 변환
# id -> 문자열
master_df['id'] = master_df['id'].astype('str')
# movieCd -> 문자열
master_df['movieCd'] = master_df['movieCd'].astype('str')

# 중복값 확인
master_df.duplicated().sum()
master_df['movieCd'].duplicated().sum()

# 결측값 확인
master_df.isna().sum()
master_df[master_df['score'].isna()]
# 단 한 건의 영화이고, 해당 영화에 대한 정보를 확보할 수 있으므로
# 수기로 결측 대체 진행

# 영화 코드가 있으므로, 해당 코드를 기반으로 검색 진행
master_df.loc[master_df['movieCd'] == '20264635',
              ['score', 'id', 'genre', 'grade', 'time', 'director', 'expert_score', 'nation']
            ] = [0, 'N/A', '애니메이션', '12세이상관람가', '109분', '하스이 타카히로', 0, '일본']
master_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         12 non-null     object 
 1   score         11 non-null     float64
 2   id            11 non-null     float64
 3   genre         11 non-null     object 
 4   grade         11 non-null     object 
 5   time          11 non-null     object 
 6   director      11 non-null     object 
 7   expert_score  11 non-null     float64
 8   nation        11 non-null     object 
 9   movieCd       12 non-null     int64  
dtypes: float64(3), int64(1), object(6)
memory usage: 1.1+ KB


,title,score,id,genre,grade,time,director,expert_score,nation,movieCd
0,눈동자,5.00,63186.0,스릴러,15세이상관람가,105분,염지호,5.00,한국,20242402
1,다윗,5.00,63108.0,애니메이션,전체관람가,109분,필커닝햄,5.00,미국,20262902
2,명탐정 코난: 하이웨이의 타천사,0.00,N/A,애니메이션,12세이상관람가,109분,하스이 타카히로,0.00,일본,20264635
3,모아나,5.67,62818.0,"어드벤처,액션",전체관람가,115분,토마스케일,5.67,미국,20259946
4,미니언즈 & 몬스터즈,6.50,63032.0,애니메이션,전체관람가,89분,피에르코팽,6.50,미국,20261784


### movie_df : KOBIS API를 통해 수집한 일별 영화 순위 및 정보

* `rank` : 순위
* `rankInten` : 이전일 대비 순위 증감분
* `rankOldAndNew` : 이전일 대비 랭크 신규 진입 여부 (OLD : 기존 / NEW : 신규)
* `movieCd` : 영화 대표 코드
* `movieNm` : 영화 이름 (국문)
* `openDt` : 영화 개봉일
* `salesAmt` : 해당 날짜의 매출액
* `salesShare` : 해당일자 상영작의 매출총액 대비 해당 영화의 매출비율
* `salesInten` : 전일 대비 매출액 증감분
* `salesChange` : 전일 대비 매출 증감 비율
* `salesAcc` : 누적 매출액
* `audiCnt` : 해당일의 관객수
* `audiInten` : 전일 대비 관객수 증감분
* `audiChange` : 전일 대비 관객수 증감 비율
* `audiAcc` : 누적관객수
* `scrnCnt` : 해당 일자에 상영한 스크린 수 출력
* `showCnt` : 해당 일자에 상영된 횟수 출력

In [23]:
# 데이터 확인
# movie_df.info()

# 데이터 형 변환
# movieCd -> 문자열
master_df['movieCd'] = master_df['movieCd'].astype('str')

# 중복값 확인
movie_df.duplicated().sum()

# 결측값 확인
movie_df.isna().sum()

# 이상치 확인
movie_df.describe()
#  누적 매출액 >= 해당일 매출액
movie_df[movie_df['salesAcc'] < movie_df['salesAmt']]
# 누적 관객수 >= 해당일 관객수
movie_df[movie_df['audiAcc'] < movie_df['audiCnt']]
# 상영 횟수 >= 상영 스크린 수
movie_df[movie_df['showCnt'] < movie_df['scrnCnt']]
# rankOldAndNew : OLD / NEW
movie_df['rankOldAndNew'].unique()

array(['OLD', 'NEW'], dtype=object)

# 2-1. 집계 기간 내 박스오피스 영화 기본 분석

## 집계 기간 내 박스오피스 순위권 내 (10위) 포함된 영화 특징 조사

In [24]:
# 국가
master_df.groupby('nation').count() # 미국(6), 한국(4), 일본(2)

# 장르
master_df.groupby('genre').count()
# 애니메이션(5)
# 다큐멘터리, 스릴러, [액션, 모험, 드라마],[액션, 스릴러, SF], [어드벤처, 액션], 코미디, [판타지, 어드벤처, 액션] (1)

# 등급
master_df.groupby('grade').count()
# 전체관람가 (5)
# 15세이상관람가 (4)
# 12세이상관람가 (3)

,title,score,id,genre,time,director,expert_score,nation,movieCd
grade,,,,,,,,,
12세이상관람가,3,3,3,3,3,3,3,3,3
15세이상관람가,4,4,4,4,4,4,4,4,4
전체관람가,5,5,5,5,5,5,5,5,5


## 기간에 따른 영화 순위 변화

In [25]:
draw_lineChart(movie_df, 'rank', '영화별 일별 박스오피스 순위 변동 추이', '순위', True)

## 기간에 따른 일일 매출액 변화

In [35]:
draw_lineChart(movie_df, 'salesAmt', '영화별 일별 박스오피스 일일 매출액 변동 추이', '일일 매출액')

## 기간에 따른 일일 관객 수 변화

In [41]:
draw_lineChart(movie_df, 'audiCnt', '영화별 일별 박스오피스 일일 관객 수 변동 추이', '일일 관객 수')

## 기간에 따른 누적 매출액 변화

In [36]:
draw_lineChart(movie_df, 'salesAcc', '영화별 일별 박스오피스 누적 매출 변동 추이', '누적 매출')

## 기간에 따른 누적 관객 수 변화

In [42]:
draw_lineChart(movie_df, 'audiAcc', '영화별 일별 박스오피스 누적 관객 수 변동 추이', '누적 관객 수')

## 기간에 따른 스크린 수

In [37]:
draw_lineChart(movie_df, 'scrnCnt', '영화별 일별 박스오피스 스크린 수 변동 추이', '스크린 수')

## 기간에 따른 상영 수

In [38]:
draw_lineChart(movie_df, 'showCnt', '영화별 일별 박스오피스 상영 수 변동 추이', '상영 수')

* 집계 기간 내 순위권 내 포함된 영화는 총 12개의 영화가 순위권 내에 진입하였다.
  * 국가별 : 미국(6개), 한국(4개), 일본(2개)
  * 카테고리 : 애니메이션 (5개), 액션 포함 (4개)
  * 등급 : 전체관람가 (5개), 15세이상관람가(4개), 12세이상관람가(3개)

* 집계 기간 내 1위에 등극한 영화는 총 3개의 영화가 집계 기간 동안 1위에 등극
  * 호프 (26년 7월 24일 ~ 26년 7월 28일)
  * 스파이더맨: 브랜드 뉴 데이 (26년 7월 29일 ~ 26년 8월 4일)
  * 오디세이 (26년 8월 5일 ~ 26년 8월 9일)

  * 기존 1위를 유지하던 호프가 스파이더맨의 개봉과 함께 1위에서 추락하기 시작하였다.
  * 또한, 스파이더맨: 브랜드 뉴 데이 역시 오디세이의 개봉과 함께 1위를 내어주고 2위를 유지하고 있는 것으로 확인된다.

* 일일 매출액과 일일 관객 수 확인
  * 10위권 내 포함된 영화 중 유의미한 관객 및 매출의 차이를 보이는 것은 1위를 한 영화
  * 일일 매출액과 일일 관객 수는 유사한 패턴을 보이고 있다.
  * 스파이더맨: 브랜드 뉴 데이는 개봉 후 맞이한 첫 주말에 폭발적인 관객 수를 확보할 수 있었다.
  * 오디세이의 경우에는 일일 매출액과 일일 관객 수의 패턴이 약간 상이한 모습을 보이고 있다.
  * 관객 수 대비 매출이 더 상승했다고 해석할 수 있는데, 오디세이의 경우에는 상대적으로 단가가 비싼 아이맥스, 4DX 등으로 더 많이 소비되었을 가능성이 크다.

* 누적 매출액과 누적 관객 수 확인
  * 압도적인 누적 매출액 : 왕과 사는 남자
  * 스파이더맨의 상승세 군체를 뛰어 넘었다.
  * 호프는 실질적 1위 효과에도 불구하고, 누적 매출이 그렇게 크지 않음을 확인할 수 있다.
  호프 개봉 당시 호프와 경쟁할만한 영화가 특별히 없었다고 판단할 수 있다.
  * 그에 비해 스파이더맨과 오디세이는 상승세로 지속적으로 누적 매출 및 관객이 상승할 것으로 판단된다.
  * 왕과 사는 남자의 누적 관객 수와 매출을 뛰어넘을 수 있을지 주목해 볼 필요가 있다.